In [ ]:
#create unique landfire_event_id
import arcpy
import os

# --- PATHS ---
project_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_deduped  = os.path.join(project_gdb, "LF_raw_fires_se_size_filtered_deduped")

target_field = "landfire_event_id"
source_field = "Event_ID"

print(f"Checking for '{target_field}' in {os.path.basename(lf_deduped)}...")

# 1. Add the text field if it doesn't already exist
existing_fields = [f.name for f in arcpy.ListFields(lf_deduped)]
if target_field not in existing_fields:
    print(f"Adding field '{target_field}'...")
    arcpy.management.AddField(
        in_table=lf_deduped,
        field_name=target_field,
        field_type="TEXT",
        field_length=50
    )

print(f"Copying values from '{source_field}' to '{target_field}'...")

try:
    # 2. Use a simple Calculate Field to map Event_ID into landfire_event_id
    arcpy.management.CalculateField(
        in_table=lf_deduped,
        field=target_field,
        expression=f"!{source_field}!",
        expression_type="PYTHON3"
    )
    print(f"SUCCESS: '{target_field}' is now fully populated.")

except Exception as e:
    print(f"Error transferring IDs: {e}")

finally:
    arcpy.management.ClearWorkspaceCache(project_gdb)

In [ ]:
#spatial intersection
import arcpy
import os

# --- GEODATABASE PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")

# --- FEATURE CLASS PATHS ---
events = os.path.join(classifire_gdb, "SEFM_events_94_24")
lf     = os.path.join(classifire_gdb, "LF_raw_fires_se_size_filtered_deduped")
pairs  = os.path.join(classifire_gdb, "SEFM_LF_pairs")  # Saving pairs to master workspace

arcpy.env.overwriteOutput = True

print("Running spatial intersection between SEFM events and LANDFIRE perimeters...")

try:
    # --- INTERSECT ---
    # Intersects SEFM events with deduplicated LANDFIRE layer, preserving all fields
    arcpy.analysis.Intersect(
        in_features=[events, lf],
        out_feature_class=pairs,
        join_attributes="ALL"
    )

    # Quick quality check
    match_count = int(arcpy.management.GetCount(pairs)[0])
    print("-" * 60)
    print(f"Intersection complete. Generated {match_count:,} overlap candidate rows in SEFM_LF_pairs.")
    print("-" * 60)

except Exception as e:
    print(f"Error during intersection process: {e}")

finally:
    # Clear database cache to manage schema locks
    arcpy.management.ClearWorkspaceCache(classifire_gdb)

In [ ]:
# ===================================================================
# Create and calculate temporal overlap flag for paired records
# ===================================================================
import arcpy
import os
from datetime import datetime, timedelta

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
lf_pairs       = os.path.join(classifire_gdb, "SEFM_LF_pairs")

target_field = "temporal_match"

# --- STEP 1: INITIALIZE THE FIELD ---
print(f"Checking for '{target_field}' field in SEFM_LF_pairs...")
existing_fields = [f.name for f in arcpy.ListFields(lf_pairs)]

if target_field not in existing_fields:
    print(f"Adding field '{target_field}'...")
    arcpy.management.AddField(
        in_table=lf_pairs,
        field_name=target_field,
        field_type="SHORT"
    )

# --- STEP 2: EVALUATE TEMPORAL OVERLAP ---
print("Evaluating date overlaps between SEFM and LANDFIRE intervals...")

# 30-day buffer
delta = timedelta(days=30)

fields = [
    "event_id",                 # SEFM Event ID
    "MIN_prebd_min_corrected",   # SEFM window start (integer YYYYMMDD)
    "MAX_bd_min_corrected_plus8",# SEFM window end (integer YYYYMMDD)
    "start_date_corrected",     # LANDFIRE corrected start (datetime)
    "end_date_corrected",       # LANDFIRE corrected end (datetime)
    "temporal_match"            # Output flag
]

# Track total matches for console feedback
match_count = 0

with arcpy.da.UpdateCursor(lf_pairs, fields) as cur:
    for row in cur:
        # Unpack variables safely to avoid scoping bugs
        eid, min_prebd, max_bd8, lf_start, lf_end, tmatch = row

        # Initialize/Fallback
        tmatch = 0

        # --- LANDFIRE interval must exist and SEFM fields must be populated ---
        if lf_start is None or lf_end is None or min_prebd is None or max_bd8 is None:
            tmatch = 0
        else:
            try:
                # --- Convert SEFM interval (YYYYMMDD integers → datetime) ---
                sefm_start = datetime.strptime(str(int(min_prebd)), "%Y%m%d")
                sefm_end   = datetime.strptime(str(int(max_bd8)), "%Y%m%d")

                # --- Expand LANDFIRE interval by ±30 days ---
                lf_start_expanded = lf_start - delta
                lf_end_expanded   = lf_end + delta

                # --- Overlap test ---
                if (sefm_start <= lf_end_expanded and sefm_end >= lf_start_expanded):
                    tmatch = 1
                    match_count += 1
                else:
                    tmatch = 0
                    
            except ValueError:
                # Catch any unexpected malformed date integers in the raw table
                tmatch = 0

        # Update the database row with the re-assembled list
        cur.updateRow([eid, min_prebd, max_bd8, lf_start, lf_end, tmatch])

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print(f"SUCCESS: Temporal evaluation complete.")
print(f"Flagged {match_count:,} records as valid temporal matches (tmatch = 1).")
print("-" * 60)

In [ ]:
# ===================================================================
# Add 'landfire_match' tracking column to master SEFM layer
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

target_field = "landfire_match"

print(f"Checking fields in {os.path.basename(events)}...")
existing_fields = [f.name for f in arcpy.ListFields(events)]

# --- ADD FIELD ---
if target_field not in existing_fields:
    print(f"Adding '{target_field}' field to master events layer...")
    arcpy.management.AddField(
        in_table=events,
        field_name=target_field,
        field_type="TEXT",
        field_length=50,
        field_alias="landfire_match"
    )
    print(f"Field '{target_field}' successfully added.")
else:
    print(f"Field '{target_field}' already exists. Skipping.")

# Clear schema locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

In [ ]:
# ===================================================================
# Identify the dominant LANDFIRE Event_Type based on maximum 
#          intersecting area and update the master events layer.
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_LF_pairs")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

# --- SCHEMA CHECK ---
pair_fields = [f.name for f in arcpy.ListFields(pairs)]
lf_event_type_field = "Event__Type" if "Event__Type" in pair_fields else "Event_Type"

# --- 1. BUILD THE DOMINANT LOOKUP DICTIONARY ---
print("Building dominance lookup based on maximum overlap area...")

# Tracking both the winning type and the max area found so far:
# Structure: { event_id: (Event_Type, max_area) }
event_dominance = {}

# We include Shape_Area in the cursor to evaluate the size of the intersection
fields = ["event_id", lf_event_type_field, "temporal_match", "Shape_Area"]

with arcpy.da.SearchCursor(pairs, fields) as cur:
    for row in cur:
        eid, lf_type, tmatch, area = row
        
        # Only process high-confidence space-time matches
        if tmatch == 1 and eid is not None and lf_type is not None:
            
            # Convert ID to string to ensure consistent dictionary keys
            eid_str = str(eid)
            
            # If we've seen this event ID before, let the larger area win
            if eid_str in event_dominance:
                current_winning_type, max_area = event_dominance[eid_str]
                
                if area > max_area:
                    # New larger intersection found; overwrite with dominant type
                    event_dominance[eid_str] = (str(lf_type), area)
            else:
                # First time seeing this event ID, store it
                event_dominance[eid_str] = (str(lf_type), area)

# Convert our tracking dictionary into a clean {event_id: dominant_type} mapping
event_to_lf = {eid: data[0] for eid, data in event_dominance.items()}

print(f"Dominance processing complete. Resolved {len(event_to_lf):,} unique master events.")

# --- 2. UPDATE THE MASTER EVENTS LAYER ---
print(f"Writing dominant LANDFIRE labels to master {os.path.basename(events)} layer...")
updated_count = 0

with arcpy.da.UpdateCursor(events, ["event_id", "landfire_match"]) as cur:
    for row in cur:
        eid = str(row[0]) if row[0] is not None else None
        
        if eid in event_to_lf:
            row[1] = event_to_lf[eid]
            updated_count += 1
        else:
            # FIXED: Setting this to None saves it as a clean database NULL
            row[1] = None  
            
        cur.updateRow(row)

# Clear locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print(f"SUCCESS. Assigned dominant LANDFIRE labels to {updated_count:,} master events.")
print("-" * 60)

In [ ]:
# ===================================================================
# Create 'sefm_match' field in LANDFIRE layer if missing, 
#          then populate it (1 = matched, None/NULL = unmatched).
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_LF_pairs")
lf_deduped     = os.path.join(classifire_gdb, "LF_raw_fires_se_size_filtered_deduped")

target_field = "sefm_match"

# --- STEP 1: INITIALIZE THE FIELD ---
print(f"Checking fields in {os.path.basename(lf_deduped)}...")
existing_fields = [f.name for f in arcpy.ListFields(lf_deduped)]

if target_field not in existing_fields:
    print(f"Adding '{target_field}' field to deduplicated LANDFIRE layer...")
    arcpy.management.AddField(
        in_table=lf_deduped,
        field_name=target_field,
        field_type="SHORT",
        field_alias="sefm_match"
    )
    print(f"Field '{target_field}' successfully added.")
else:
    print(f"Field '{target_field}' already exists. Skipping initialization.")


# --- STEP 2: BUILD SET OF CONFIRMED MATCHED LANDFIRE IDs ---
print("Scanning pairs table for successful LANDFIRE temporal matches...")
matched_lf_ids = set()

with arcpy.da.SearchCursor(pairs, ["landfire_event_id", "temporal_match"]) as search_cur:
    for lf_id, tmatch in search_cur:
        if tmatch == 1 and lf_id is not None:
            matched_lf_ids.add(str(lf_id))

print(f"Lookup set complete. Found {len(matched_lf_ids):,} unique LANDFIRE fires captured by SEFM.")


# --- STEP 3: UPDATE THE MASTER LANDFIRE LAYER ---
print(f"Writing validation flags to {os.path.basename(lf_deduped)}...")
matched_count   = 0
unmatched_count = 0

with arcpy.da.UpdateCursor(lf_deduped, ["landfire_event_id", "sefm_match"]) as update_cur:
    for row in update_cur:
        lf_id = str(row[0]) if row[0] is not None else None
        
        if lf_id in matched_lf_ids:
            row[1] = 1
            matched_count += 1
        else:
            # Replicated MTBS logic: Unmatched entries are left as clean database NULLs
            row[1] = None  
            unmatched_count += 1
            
        update_cur.updateRow(row)

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print("SUCCESS: LANDFIRE match tracking update complete.")
print(f"Total Fires Successfully Captured (1): {matched_count:,}")
print(f"Total Fires Missed by SEFM    (NULL): {unmatched_count:,}")
print("-" * 60)

In [ ]:
# ===================================================================
# SCRIPT STEP: LF_SEFM_09_Append_LF_Dates_Direct.py
# PURPOSE: Extract start_date_corrected from pairs table where 
#          temporal_match == 1 and apply directly to master layer.
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_LF_pairs")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

target_field = "landfire_start_date"

# --- STEP 1: VERIFY FIELD EXISTENCE ---
existing_fields = [f.name for f in arcpy.ListFields(events)]
if target_field not in existing_fields:
    print(f"Adding true DATE field '{target_field}'...")
    arcpy.management.AddField(events, target_field, "DATE", field_alias="landfire_start_date")
else:
    print(f"Field '{target_field}' exists. Overwriting with direct pairs mapping...")

# --- STEP 2: BUILD DICTIONARY DRIVEN STRICTLY BY TEMPORAL_MATCH == 1 ---
print("Extracting dates from pairs table where temporal_match == 1...")
event_to_date_lookup = {}

with arcpy.da.SearchCursor(pairs, ["event_id", "start_date_corrected", "temporal_match"]) as search_cur:
    for eid, lf_start, tmatch in search_cur:
        eid_str = str(eid)
        if tmatch == 1 and eid_str is not None and lf_start is not None:
            event_to_date_lookup[eid_str] = lf_start

print(f"Successfully resolved dates for {len(event_to_date_lookup):,} verified LANDFIRE pairs.")

# --- STEP 3: UPDATE THE MASTER SEFM LAYER DIRECTLY ---
print(f"Writing validated LANDFIRE start dates to master SEFM layer...")
updated_count = 0
cleared_count = 0

with arcpy.da.UpdateCursor(events, ["event_id", target_field]) as update_cur:
    for row in update_cur:
        sefm_id = str(row[0]) if row[0] is not None else None
        
        # Bypass tracking column checks: If the ID proved it had a match in Step 2, update it!
        if sefm_id in event_to_date_lookup:
            row[1] = event_to_date_lookup[sefm_id]
            updated_count += 1
        else:
            row[1] = None  
            cleared_count += 1
            
        update_cur.updateRow(row)

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print(f"SUCCESS: Master SEFM layer updated for '{target_field}'.")
print(f"Correctly populated dates: {updated_count:,}")
print(f"Rows left as NULL (unmatched): {cleared_count:,}")
print("-" * 60)

In [ ]:
# ===================================================================
# Extract end_date_corrected from pairs table where 
#          temporal_match == 1 and apply directly to master layer.
# ===================================================================
import arcpy
import os

# --- PATHS ---
classifire_gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
pairs          = os.path.join(classifire_gdb, "SEFM_LF_pairs")
events         = os.path.join(classifire_gdb, "SEFM_events_94_24")

target_field = "landfire_end_date"

# --- STEP 1: VERIFY FIELD EXISTENCE ---
print(f"Checking fields in {os.path.basename(events)}...")
existing_fields = [f.name for f in arcpy.ListFields(events)]

if target_field not in existing_fields:
    print(f"Adding true DATE field '{target_field}'...")
    arcpy.management.AddField(
        in_table=events,
        field_name=target_field,
        field_type="DATE",
        field_alias="landfire_end_date"
    )
    print(f"Field '{target_field}' successfully added.")
else:
    print(f"Field '{target_field}' already exists. Overwriting with direct pairs mapping...")

# --- STEP 2: BUILD DICTIONARY DRIVEN STRICTLY BY TEMPORAL_MATCH == 1 ---
print("Extracting end dates from pairs table where temporal_match == 1...")
event_to_date_lookup = {}

# Pull event_id and end_date_corrected from high-confidence rows
with arcpy.da.SearchCursor(pairs, ["event_id", "end_date_corrected", "temporal_match"]) as search_cur:
    for eid, lf_end, tmatch in search_cur:
        eid_str = str(eid)
        if tmatch == 1 and eid_str is not None and lf_end is not None:
            event_to_date_lookup[eid_str] = lf_end

print(f"Successfully resolved end dates for {len(event_to_date_lookup):,} verified LANDFIRE pairs.")

# --- STEP 3: UPDATE THE MASTER SEFM LAYER DIRECTLY ---
print(f"Writing validated LANDFIRE end dates to master SEFM layer...")
updated_count = 0
cleared_count = 0

with arcpy.da.UpdateCursor(events, ["event_id", target_field]) as update_cur:
    for row in update_cur:
        sefm_id = str(row[0]) if row[0] is not None else None
        
        # Directly match based on the verified pairings dictionary
        if sefm_id in event_to_date_lookup:
            row[1] = event_to_date_lookup[sefm_id]
            updated_count += 1
        else:
            row[1] = None  # Force unmatched records to a clean database NULL
            cleared_count += 1
            
        update_cur.updateRow(row)

# Clear database memory locks
arcpy.management.ClearWorkspaceCache(classifire_gdb)

print("-" * 60)
print(f"SUCCESS: Master SEFM layer updated for '{target_field}'.")
print(f"Correctly populated dates: {updated_count:,}")
print(f"Rows left as NULL (unmatched): {cleared_count:,}")
print("-" * 60)